# Quantum Solvers for Predictive Aerodynamic Modeling

*2026 Global Quantum + AI Challenge*

Accurately modeling aerodynamics is critical for peak design efficiency in aerospace.
Classical CFD solvers face computational bottlenecks when simulating turbulent flow
around complex geometries — millions of mesh points, thousands of design iterations.

This notebook demonstrates how uniqx accelerates aerodynamic simulation using
**Lattice Boltzmann Methods (LBM)** compiled to hybrid hardware through uniqx:

| Workload | Operation | Hardware affinity |
|:---------|:----------|:-----------------:|
| Wing cross-section flow | Dense collision + streaming (`matmul`) | **GPU** |
| Thermal boundary layer | Matrix exponential diffusion (`expv`) | **GPU / QPU** |
| Turbulence cascade analysis | Eigendecomposition (`eigs`) | **QPU** |

**Challenge**: Can quantum-enhanced solvers radically boost aerodynamic design capabilities?

All computation is server-side. uniqx's cost model routes each workload optimally.

In [ ]:
import os
from uniqx import connect

# Default to a local service for development.
# For prod, export UNIQX_GATEWAY=api.oriqx.com:443 and UNIQX_API_KEY=...
endpoint = os.environ.get("UNIQX_GATEWAY", "localhost:50050")
client = connect(endpoint)
print("Connected to", endpoint)
import time
import numpy as np
import matplotlib.pyplot as plt

from uniqx.domains.physics.cfd import (
    ChannelFlow,
    LidDrivenCavity,
    build_lbm_step_module,
    build_diffusion_step_module,
)
from uniqx import ops, tracing, parse_result
from uniqx.core.execution import connect, preflight, submit, get, fmt_mat, fmt_vec

API_KEY = os.environ.get("UNIQX_API_KEY")


## 1. Airfoil Flow Setup

We configure wing cross-section simulations at three Reynolds numbers representing
distinct flight regimes. Higher Re means finer turbulence structures and larger
operator matrices — exactly the regime where GPU and QPU acceleration pays off.

| Regime | Re | Grid | Physics |
|:-------|---:|:-----|:--------|
| Cruise | 200 | 16×8 | Laminar, attached flow |
| Takeoff | 500 | 16×8 | Transitional, high-lift |
| Turbulent | 1000 | 20×10 | Fully turbulent, separation |

In [ ]:
cruise = ChannelFlow(nx=16, ny=8, Re=200)
takeoff = ChannelFlow(nx=16, ny=8, Re=500)
turbulent = ChannelFlow(nx=20, ny=10, Re=1000)

regimes = [
    ("cruise", cruise),
    ("takeoff", takeoff),
    ("turbulent", turbulent),
]

for name, flow in regimes:
    print(f"{name.upper()}:")
    print(f"  Grid: {flow.nx} x {flow.ny} = {flow.dim} nodes")
    print(f"  Re={flow.Re}, tau={flow.tau:.4f}, nu={flow.nu:.6f}")
    print(f"  LBM state vector: {flow.dim * 9} elements (9 velocities per node)")
    print(f"  Operator matrix: {flow.dim * 9} x {flow.dim * 9} = {(flow.dim * 9) ** 2} entries")
    print()

## 2. Trace Aerodynamic Modules

Build LBM modules for each flight regime. The `build_lbm_step_module` function
traces the combined collision + streaming operator $A = S \cdot C$ as a single
`matmul` primitive. uniqx compiles this to the optimal hardware backend.

In [ ]:
modules = {}

for name, flow in regimes:
    mod, inputs, meta = build_lbm_step_module(flow)
    modules[name] = {"mod": mod, "inputs": inputs, "meta": meta}
    print(f"{name:<12} dim={meta['dim']:>4}, state_size={meta['state_size']:>5}, IR={len(mod.to_text()):>6} chars")

## 3. Preflight: Hardware Routing

uniqx analyses each module and returns Pareto-optimal execution options.
For the dense LBM collision matrix multiply, GPU parallelism dominates at scale.
We compare all three regimes to see how hardware preferences shift with problem size.

In [ ]:
def print_options(label, options):
    print(f"\n--- {label} ---")
    if not options:
        print("  (no options)")
        return
    print(
        f"  {'Label':<24} {'Time':>10} {'Cost ($)':>12} {'Error':>8} {'Carbon (g)':>11}"
    )
    print(f"  {'-' * 24} {'-' * 10} {'-' * 12} {'-' * 8} {'-' * 11}")
    for opt in options:
        flag = "  <-- rec" if opt.get("recommended") else ""
        print(
            f"  {opt['label']:<24} {opt['total_time']:>10.2f}"
            f"  ${opt['total_cost_usd']:>10.6f}"
            f"  {opt['max_error_rate']:>8.4f}"
            f"  {opt['total_carbon_g']:>10.3f}{flag}"
        )


preflight_results = {}

for name, flow in regimes:
    m = modules[name]
    opts = preflight(m["mod"], client=client)
    preflight_results[name] = opts
    print_options(
        f"{name.capitalize()} (Re={flow.Re}, {flow.nx}x{flow.ny}, dim={m['meta']['dim']})",
        opts,
    )

## 4. Execute Flow Simulations

Run all three flight regimes through uniqx with each available hardware option.
We use iterative mode to simulate multiple LBM time steps in a single uniqx job,
collecting kinetic energy as a convergence diagnostic.

In [ ]:
N_STEPS = 8
results = {}

for name, flow in regimes:
    m = modules[name]
    opts = preflight_results[name]
    results[name] = {}

    for opt in opts:
        label = opt["label"]
        print(f"\n{name} / {label}:")

        t0 = time.monotonic()
        jid = submit(
            m["mod"],
            client=client,
            runtime_inputs=m["inputs"] + [f"iterations={N_STEPS}"],
            preflight_job_id=opts.job_id,
            option_idx=opt["_idx"],
        )
        res = get(jid, client=client)
        wall = time.monotonic() - t0

        payload = res.get("payload", b"")
        if isinstance(payload, str):
            payload = payload.encode()
        out = parse_result(payload, ["final_state", "energy"])

        energy = out.get("energy", ([], "", [0.0]))[2][0] if out else 0.0
        print(f"  {wall:.2f}s, kinetic energy = {energy:.6f}")

        results[name][label] = {
            "time": wall,
            "energy": energy,
            "option": opt,
        }

## 5. Thermal Boundary Layer Analysis

Heat transfer around wing surfaces is governed by the thermal boundary layer.
We model this as 2D heat diffusion solved via matrix exponential (`expv`).
Increasing resolution captures thinner boundary layers at higher Re — a workload
where QPU-accelerated matrix exponential becomes advantageous.

In [ ]:
boundary_sizes = [(12, 12), (20, 20)]
diff_results = {}

for nx, ny in boundary_sizes:
    mod_d, inputs_d, meta_d = build_diffusion_step_module(
        nx, ny, alpha=0.01, n_steps=8
    )
    opts_d = preflight(mod_d, client=client)
    print_options(f"Boundary Layer {nx}x{ny} (dim={meta_d['dim']})", opts_d)

    diff_results[(nx, ny)] = {}
    for opt in opts_d:
        t0 = time.monotonic()
        jid = submit(
            mod_d,
            client=client,
            runtime_inputs=inputs_d,
            preflight_job_id=opts_d.job_id,
            option_idx=opt["_idx"],
        )
        res = get(jid, client=client)
        wall = time.monotonic() - t0
        diff_results[(nx, ny)][opt["label"]] = {"time": wall, "option": opt}
        print(f"  {opt['label']}: {wall:.2f}s")

## 6. Turbulence Cascade Analysis

The LBM step operator $A = S \cdot C$ encodes the linearised dynamics of the flow.
Its eigenvalues reveal **hydrodynamic modes**: the dominant eigenmodes correspond
to the slowest-decaying flow structures (large-scale vortices), while fast-decaying
modes represent the turbulent energy cascade to small scales.

We use `ops.eigs` on the combined LBM operator to extract the $k$ dominant
eigenvalues — a stability analysis that uniqx can route to QPU hardware
for eigendecomposition at scale.

In [ ]:
from uniqx.domains.physics.cfd import _build_combined_step


@tracing.to_module(name="turbulence_modes")
def turbulence_modes(operator):
    """Find the dominant hydrodynamic modes of the LBM step operator.

    The largest-magnitude eigenvalues correspond to the slowest-decaying
    flow structures (large-scale vortices and mean-flow modes).
    """
    eigenvalues, eigenvectors = ops.eigs(operator, k=4, hermitian=False, which="largest")
    return eigenvalues, eigenvectors


# Build the combined LBM operator for the cruise regime
turb_flow = ChannelFlow(nx=12, ny=6, Re=200)
A_turb = _build_combined_step(
    turb_flow.nx, turb_flow.ny, turb_flow.tau, periodic_x=True, periodic_y=False
)
state_size = turb_flow.dim * 9

print(f"Turbulence analysis:")
print(f"  Grid: {turb_flow.nx}x{turb_flow.ny} = {turb_flow.dim} nodes")
print(f"  Operator: {state_size}x{state_size} ({state_size ** 2} entries)")

# Trace the eigendecomposition module
mod_turb = turbulence_modes(A_turb)
print(f"  IR size: {len(mod_turb.to_text())} chars")

# Preflight
opts_turb = preflight(mod_turb, client=client)
print_options(f"Turbulence Modes ({turb_flow.nx}x{turb_flow.ny})", opts_turb)

# Execute with each available option
turb_inputs = [fmt_mat(A_turb, state_size, state_size)]

for opt in opts_turb:
    label = opt["label"]
    t0 = time.monotonic()
    jid = submit(
        mod_turb,
        client=client,
        runtime_inputs=turb_inputs,
        preflight_job_id=opts_turb.job_id,
        option_idx=opt["_idx"],
    )
    res = get(jid, client=client)
    wall = time.monotonic() - t0

    payload = res.get("payload", b"")
    if isinstance(payload, str):
        payload = payload.encode()
    out = parse_result(payload, ["eigenvalues", "eigenvectors"])

    eigs_raw = out.get("eigenvalues", ([], "", []))[2] if out else []
    print(f"\n  {label}: {wall:.2f}s")
    if eigs_raw:
        print(f"  Dominant eigenvalues (magnitude): {[abs(e) for e in eigs_raw[:4]]}")
        print(f"  Spectral radius: {max(abs(e) for e in eigs_raw):.6f}")

## 7. Design Iteration Scaling

In aircraft design, thousands of geometry variants must be evaluated.
How does hardware choice change with problem size? We sweep grid resolutions
and collect preflight estimates to reveal the crossover where GPU and QPU
acceleration becomes cost-effective.

In [ ]:
grid_sizes = [(8, 4), (16, 8), (24, 12)]
scaling = []

for nx, ny in grid_sizes:
    prob = ChannelFlow(nx=nx, ny=ny, Re=200)
    mod_s, inp_s, meta_s = build_lbm_step_module(prob)
    opts_s = preflight(mod_s, client=client)
    dim = meta_s["dim"]
    print(f"\n{nx}x{ny} (dim={dim}): {len(opts_s)} options")
    for opt in opts_s:
        flag = " *" if opt.get("recommended") else ""
        print(
            f"  {opt['label']:<20} time={opt['total_time']:>8.1f}  cost=${opt['total_cost_usd']:.4f}{flag}"
        )
        scaling.append(
            {
                "nx": nx,
                "ny": ny,
                "dim": dim,
                "label": opt["label"],
                "time": opt["total_time"],
                "cost": opt["total_cost_usd"],
                "recommended": opt.get("recommended", False),
            }
        )

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors_hw = {"cpu-only": "#2563eb", "cpu+gpu": "#16a34a", "cpu+gpu+qpu": "#9333ea"}

# Top-left: Execution time by flight regime (grouped bars)
all_labels = sorted(set(l for n in results for l in results[n]))
x = np.arange(len(all_labels))
width = 0.25
regime_names = ["cruise", "takeoff", "turbulent"]
regime_colors = ["#2563eb", "#16a34a", "#dc2626"]
for i, name in enumerate(regime_names):
    times = [results[name].get(l, {}).get("time", 0) for l in all_labels]
    axes[0, 0].bar(x + i * width, times, width, label=name, color=regime_colors[i], alpha=0.8)
axes[0, 0].set_xticks(x + width)
axes[0, 0].set_xticklabels(all_labels, fontsize=8)
axes[0, 0].set_ylabel("Wall time (s)")
axes[0, 0].set_title(f"Airfoil LBM Execution ({N_STEPS} steps)")
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(axis="y", alpha=0.3)

# Top-right: Boundary layer diffusion scaling (line plot)
for label_key in sorted(set(l for sz in diff_results for l in diff_results[sz])):
    dims = []
    times = []
    for sz in sorted(diff_results.keys()):
        if label_key in diff_results[sz]:
            dims.append(sz[0] * sz[1])
            times.append(diff_results[sz][label_key]["time"])
    c = colors_hw.get(label_key, "#6b7280")
    axes[0, 1].plot(dims, times, "o-", color=c, label=label_key)
axes[0, 1].set_xlabel("Grid points")
axes[0, 1].set_ylabel("Wall time (s)")
axes[0, 1].set_title("Thermal Boundary Layer (expv) Scaling")
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(alpha=0.3)

# Bottom-left: Preflight time scaling (semilogy)
hw_labels = sorted(set(d["label"] for d in scaling))
for hw in hw_labels:
    sub = [d for d in scaling if d["label"] == hw]
    c = colors_hw.get(hw, "#6b7280")
    axes[1, 0].semilogy(
        [d["dim"] for d in sub], [d["time"] for d in sub], "o-", color=c, label=hw
    )
axes[1, 0].set_xlabel("Nodes (nx*ny)")
axes[1, 0].set_ylabel("Estimated time (log)")
axes[1, 0].set_title("Design Iteration: Preflight Time Scaling")
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(alpha=0.3)

# Bottom-right: Cost scaling
for hw in hw_labels:
    sub = [d for d in scaling if d["label"] == hw]
    c = colors_hw.get(hw, "#6b7280")
    axes[1, 1].plot(
        [d["dim"] for d in sub], [d["cost"] for d in sub], "s-", color=c, label=hw
    )
axes[1, 1].set_xlabel("Nodes (nx*ny)")
axes[1, 1].set_ylabel("Estimated cost (USD)")
axes[1, 1].set_title("Design Iteration: Preflight Cost Scaling")
axes[1, 1].legend(fontsize=8)
axes[1, 1].grid(alpha=0.3)

fig.suptitle("Aerodynamic Modeling on Hybrid Hardware", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Summary

| Workload | Key op | Small grid | Large grid |
|:---------|:-------|:-----------|:-----------|
| Airfoil LBM step | `matmul` (dense) | CPU (low overhead) | GPU (parallelism) |
| Thermal boundary layer | `expv` (matrix exp) | CPU | GPU / QPU |
| Turbulence modes | `eigs` (eigendecomp) | GPU | QPU (polynomial scaling) |

**Key results:**

1. **Flight regime comparison** — Turbulent flow (Re=1000) benefits most from GPU/QPU
 acceleration due to the larger operator matrices.
2. **Boundary layer resolution** — Finer thermal grids shift the cost-optimal hardware
 from CPU to GPU, and eventually to QPU as matrix exponential scales.
3. **Turbulence spectrum** — Eigendecomposition of the LBM operator reveals dominant
 hydrodynamic modes. QPU routing enables analysis of larger meshes where classical
 eigensolvers become prohibitive.
4. **Design iteration scaling** — At production mesh densities, uniqx's cost model
 automatically routes to hybrid CPU+GPU+QPU execution, enabling thousands of design
 variants within practical time and cost budgets.

uniqx's cost model automatically selects the right hardware at each scale.
Users submit the same traced module regardless of backend — the platform handles routing.